# Natural Language Processing Project - Menganalisis Dampak Terhadap Nilai Tukar Dolar
Arranged by:
*   Anders Emmanuel Tan (24/541351/PA/22964)
*   Azhar Maulana (24/533487/PA/22582)
*   Evan Razzan Adytaputra (24/545257/PA/23166)
*   Kukuh Agus Hermawan (24/533395/PA/22573)

Deskripsi Modul: Notebook ini berfungsi sebagai pengumpul data berita geopolitik secara otomatis.

## 0. Instalasi Library pendukung
Description: Cell dibawah ini berfungsi untuk menginstal beberapa library Python yang dibutuhkan untuk *web scraping*, mengambil berita, parsing HTML, dan memproses data.

In [11]:
%pip install -q trafilatura googlenewsdecoder beautifulsoup4 pandas requests

Note: you may need to restart the kernel to use updated packages.


## 1. Pengaturan Parameter Scraping dan Penyaring Kata Kunci

Bagian ini mengatur bagaimana pengambilan berita akan dijalankan:
* **Rentang Waktu**: Mengambil data dari September 2021 hingga September 2026.
* **Interval**: Mengambil data setiap 2 hari sekali.
* **Penyaringan Kata Kunci (Filtering)**:
  * **Diabaikan (`EXCLUDE_KEYWORDS`)**: Menghapus berita seputar hiburan, kripto, gaya hidup, atau tips keuangan pribadi agar data tetap fokus pada geopolitik.
  * **Diutamakan (`CORE_GEOPOLITICAL_KEYWORDS`)**: Hanya menyimpan berita yang mengandung isu geopolitik, kebijakan bank sentral, suku bunga, atau perang.

In [12]:
import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import trafilatura
from googlenewsdecoder import gnewsdecoder
from bs4 import BeautifulSoup

START_DATE = datetime(2021, 9, 1)
END_DATE = datetime(2026, 9, 1)

SAMPLE_INTERVAL_DAYS = 2     # Daily intervals for uniform temporal distribution
MAX_PER_INTERVAL = 100

OUTPUT_DIR = os.path.join("..", "data/raw")
OUTPUT_CSV = "geopolitical_news.csv"
SAVE_PATH = os.path.join(OUTPUT_DIR, OUTPUT_CSV)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Strategic Filtering: Drop consumer finance, crypto, entertainment, and lifestyle
EXCLUDE_KEYWORDS = [
    "tips for", "how to beat", "how to save", "credit score", "mortgage rate",
    "retirement", "undervalued", "nft", "meme", "doge", "crypto", "bitcoin",
    "ethereum", "token", "celebrity", "dating", "wedding", "mansion", "movie",
    "box office", "horoscope", "diet", "recipe"
]

# Strategic Filtering: Require sovereign macroeconomic and geopolitical signals[cite: 1]
CORE_GEOPOLITICAL_KEYWORDS = [
    # Core Geopolitics, Defense & Conflict
    "geopolitics", "war", "military", "defense", "security", "missile", 
    "nuclear", "nato", "taiwan", "russia", "ukraine", "china", "beijing", 
    "kremlin", "iran", "israel", "middle east", "ceasefire", "blockade",
    
    # Economic Statecraft & Trade Policy
    "sanctions", "tariff", "trade war", "embargo", "export control", 
    "opec", "oil", "energy", "brics", "g20",
    
    # Macro & FX Transmission Channels
    "dollar", "currency", "forex", "fed", "federal reserve", "central bank", 
    "interest rate", "inflation", "treasury", "yield", "debt", "bank indonesia"
]

print(f"Configuration set. Target file: {SAVE_PATH}")

Configuration set. Target file: ..\data/raw\geopolitical_news.csv


## 3. Fungsi Pembantu Ekstraksi Isi Berita dan Penyaring Otomatis

Di sini dibuat dua fungsi utama:
1. `parse_body()`: Mengambil teks utama/artikel dari halaman web HTML secara bersih dan membuang elemen yang tidak penting.
2. `passes_strategic_filter()`: Memeriksa apakah suatu berita layak disimpan. Berita akan dibuang jika mengandung kata-kata yang diabaikan dan hanya diterima jika memiliki minimal 1 indikator geopolitik/ekonomi utama pada judul atau paragraf awal.

In [13]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
})

import re

def compile_keyword_patterns(keywords):
    """Compiles keywords into word-boundary regex patterns.
    Prevents substring false-matches, e.g. 'war' inside 'Warriors',
    or 'dating' inside 'updating'."""
    return [re.compile(r'\b' + re.escape(kw) + r'\b', re.IGNORECASE) for kw in keywords]

EXCLUDE_PATTERNS = compile_keyword_patterns(EXCLUDE_KEYWORDS)
CORE_PATTERNS = compile_keyword_patterns(CORE_GEOPOLITICAL_KEYWORDS)

def matches_any(patterns, text: str) -> bool:
    """Returns True if any compiled word-boundary pattern is found in text."""
    return any(p.search(text) for p in patterns)

def parse_body(html_text: str) -> str:
    """Extracts readable article content with fallback to standard paragraph tags."""
    text = trafilatura.extract(html_text, include_comments=False, include_tables=False)
    if text and len(text.strip()) > 150:
        return text.strip()
    soup = BeautifulSoup(html_text, "html.parser")
    paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 40]
    extracted = " ".join(paragraphs)
    return extracted.strip() if len(extracted) > 150 else ""

def passes_strategic_filter(title: str, content: str) -> bool:
    """Applies negative noise suppression and positive macroeconomic relevance gating,
    using word-boundary matching to avoid substring false positives/negatives."""
    title_lower = title.lower()
    content_lower = content.lower()
    full_text = title_lower + " " + content_lower

    # 1. Negative Noise Filtering (word-boundary safe)
    if matches_any(EXCLUDE_PATTERNS, title_lower):
        return False

    # 2. Positive Filtering on title and lead text (word-boundary safe)
    matched_signals = sum(1 for p in CORE_PATTERNS if p.search(full_text))
    return matched_signals >= 1

## 4. Sistem Checkpoint

Kode ini memeriksa apakah sudah ada file CSV dari proses *scraping* sebelumnya:
* Jika file sudah ada, sistem akan membaca berita yang sudah pernah diunduh agar **tidak terjadi duplikasi**.
* Sistem akan melewati tanggal-tanggal yang sudah selesai dan melanjutkan *scraping* untuk tanggal yang belum terproses.
Hal tersebut dilakukan agar pengembang tidak harus mengulang proses scraping jika koneksi internet hilang.

In [14]:
# Cell 4: Check for previous CSV and identify completed checkpoints
CHECKPOINT_TRACKER = os.path.join(OUTPUT_DIR, "completed_checkpoints.txt")

checkpoints = []
curr = START_DATE
while curr <= END_DATE:
    checkpoints.append(curr)
    curr += timedelta(days=SAMPLE_INTERVAL_DAYS)

# Load actual queried dates
if os.path.exists(CHECKPOINT_TRACKER):
    with open(CHECKPOINT_TRACKER, "r") as f:
        completed_checkpoints = set(line.strip() for line in f if line.strip())
else:
    completed_checkpoints = set()

if os.path.exists(SAVE_PATH):
    df_existing = pd.read_csv(SAVE_PATH)
    processed_titles = set(df_existing["title"].dropna().tolist())
    records = df_existing.to_dict("records")
    print(f"Resuming previous progress: Loaded {len(records)} articles from {SAVE_PATH}.")
else:
    processed_titles = set()
    records = []
    print("No previous progress found. Starting from scratch.")

print(f"Total checkpoints: {len(checkpoints)} | Completed: {len(completed_checkpoints)}")

Resuming previous progress: Loaded 10653 articles from ..\data/raw\geopolitical_news.csv.
Total checkpoints: 914 | Completed: 709


## 5. Pelaksanaan Scraping Berita

Proses pengambilan data utama dilakukan pada bagian ini:
* **Multi-threading**: Menggunakan 16 *worker* sekaligus untuk mempercepat proses *download* artikel.
* **Pencarian Berita**: Mengambil berita dari Google News RSS dengan sumber yang dicantumkan ditugas (seperti CNBC).
* **Penyimpanan Otomatis**: Setiap kali satu titik waktu selesai diproses, hasilnya langsung disimpan ke dalam file CSV agar data tidak hilang jika koneksi terputus di tengah jalan.

In [15]:
# Cell 5: Fault-tolerant multi-threaded scraping with rate-limit recovery
from concurrent.futures import as_completed, ThreadPoolExecutor
import os
import random
import time
import xml.etree.ElementTree as ET
import pandas as pd
import requests

# Rejection patterns for non-article hubs and navigation dumps
INVALID_URL_PATTERN = re.compile(
    r"/video/|/tv/|/live-tv/|/author/|link\.cnbc\.com|/pro/", flags=re.IGNORECASE
)
NAVIGATION_TRASH_PATTERN = re.compile(
    r"^\s*(?:skip navigation|watch now)\b", flags=re.IGNORECASE
)


class DecoderRateLimitError(Exception):
  """Raised when Google blocks URL decoding with 429/CAPTCHA."""

  pass


def process_single_item(item, dt_start, dt_end):
  """Processes, decodes, and parses a single news item."""
  title_full = item.find("title").text if item.find("title") is not None else ""
  rss_link = item.find("link").text if item.find("link") is not None else ""
  pub_date = (
      item.find("pubDate").text if item.find("pubDate") is not None else ""
  )
  source_elem = item.find("source")
  source_name = source_elem.text if source_elem is not None else "Unknown"

  # 1. Title sanitization and title checks
  title_clean = (
      title_full.rsplit(" - ", 1)[0] if " - " in title_full else title_full
  )
  if not title_clean or title_clean in processed_titles:
    return None
  if matches_any(EXCLUDE_PATTERNS, title_clean):
    return None

  # 2. Date parsing with 1-day temporal buffer
  try:
    dt_parsed = pd.to_datetime(pub_date)
  except Exception:
    dt_parsed = dt_start

  dt_parsed_tz_naive = (
      dt_parsed.tz_localize(None)
      if getattr(dt_parsed, "tzinfo", None)
      else dt_parsed
  )
  if dt_parsed_tz_naive < (dt_start - pd.Timedelta(days=1)) or (
      dt_parsed_tz_naive > (dt_end + pd.Timedelta(days=2))
  ):
    return None

  # 3. Resolve redirected Google News URL with rate-limit detection
  actual_url = rss_link
  try:
    decoded = gnewsdecoder(rss_link, interval=0.2)
    if isinstance(decoded, dict):
      if decoded.get("status"):
        actual_url = decoded.get("decoded_url", rss_link)
      else:
        msg = str(decoded.get("message", ""))
        if "429" in msg or "sorry/index" in msg or "Too Many Requests" in msg:
          raise DecoderRateLimitError("Google 429 Rate-Limit on Decoder")
  except DecoderRateLimitError:
    raise
  except Exception:
    pass

  # Immediate non-article URL drop
  if INVALID_URL_PATTERN.search(actual_url):
    return None

  # 4. Extract clean article body
  content_text = ""
  try:
    art_resp = session.get(actual_url, timeout=6)
    if art_resp.status_code == 200:
      content_text = parse_body(art_resp.text)
  except Exception:
    pass

  # Drop navigation dumps and watch-now videos
  if not content_text or NAVIGATION_TRASH_PATTERN.match(content_text):
    return None

  # 5. Length and strategic relevance validation
  if len(content_text) >= 500 and passes_strategic_filter(
      title_clean, content_text
  ):
    return {
        "published_at": dt_parsed,
        "title": title_clean,
        "content": content_text,
        "source_domain": source_name,
        "language": "en",
        "url": actual_url,
    }
  return None


# Broadened RSS query capturing geopolitical conflict and macroeconomic channels
base_query = (
    "dollar OR currency OR Fed OR inflation OR tariff OR sanctions OR war OR"
    " military OR defense OR security OR economy OR oil"
)

idx = 1
while idx <= len(checkpoints):
  dt_start = checkpoints[idx - 1]
  dt_end = dt_start + timedelta(days=SAMPLE_INTERVAL_DAYS)
  after_str = dt_start.strftime("%Y-%m-%d")
  before_str = dt_end.strftime("%Y-%m-%d")

  # Skip if already completed in previous run
  if after_str in completed_checkpoints:
    idx += 1
    continue

  query = (
      f"({base_query}) (site:cnbc.com) after:{after_str} before:{before_str}"
  )
  rss_url = f"https://news.google.com/rss/search?q={requests.utils.quote(query)}&hl=en-US&gl=US&ceid=US:en"

  try:
    resp = session.get(rss_url, timeout=12)

    # Detect Google RSS 429 blocks
    if resp.status_code == 429 or "sorry/index" in resp.url:
      print(f"\n[!] Google RSS blocked IP on {after_str} (HTTP 429).")
      input(
          "[PAUSED] Toggle your Wi-Fi / Airplane mode to get a new IP, then"
          " press ENTER to resume..."
      )
      continue

    if resp.status_code != 200:
      print(
          f"[{idx}/{len(checkpoints)}] RSS Error HTTP {resp.status_code}. Short"
          " pause..."
      )
      time.sleep(3)
      continue

    root = ET.fromstring(resp.content)
    items = root.findall(".//item")[:MAX_PER_INTERVAL]

    if len(items) == 0:
      print(
          f"[{idx}/{len(checkpoints)}] {after_str}: Empty RSS response (0 items"
          " returned)."
      )
      input(
          "[PAUSED] Likely Google shadow-ban. Switch/reconnect network, then"
          " press ENTER to retry..."
      )
      continue

    new_articles = []
    decoder_blocked = False

    with ThreadPoolExecutor(max_workers=16) as executor:
      futures = [
          executor.submit(process_single_item, it, dt_start, dt_end)
          for it in items
      ]
      for future in as_completed(futures):
        try:
          result = future.result()
          if result and result["title"] not in processed_titles:
            new_articles.append(result)
            processed_titles.add(result["title"])
        except DecoderRateLimitError:
          decoder_blocked = True

    # If Google decoder blocked requests during processing, pause for IP rotation
    if decoder_blocked:
      print(f"\n[!] Google decoder returned 429 (Too Many Requests) on items in {after_str}.")
      input(
          "[PAUSED] Change your IP (toggle Airplane Mode/Wi-Fi hotspot), then"
          " press ENTER to retry this checkpoint..."
      )
      continue

    if new_articles:
      records.extend(new_articles)
      batch_df = pd.DataFrame(new_articles)
      header_needed = (
          not os.path.exists(SAVE_PATH) or os.path.getsize(SAVE_PATH) == 0
      )
      batch_df.to_csv(SAVE_PATH, mode="a", index=False, header=header_needed)

      print(
          f"[{idx}/{len(checkpoints)}] {after_str} to {before_str}: Saved"
          f" {len(new_articles)} new articles (Total in CSV: {len(records)})"
      )
      completed_checkpoints.add(after_str)
      with open(CHECKPOINT_TRACKER, "a") as f:
        f.write(f"{after_str}\n")
      idx += 1

    elif len(items) > 0:
      # True Legitimate Zero (feed succeeded and decoded, but items didn't match filters)
      print(
          f"[{idx}/{len(checkpoints)}] {after_str} to {before_str}: 0 relevant"
          f" articles (filtered from {len(items)} raw items)."
      )
      completed_checkpoints.add(after_str)
      with open(CHECKPOINT_TRACKER, "a") as f:
        f.write(f"{after_str}\n")
      idx += 1

  except Exception as e:
    print(f"[{idx}/{len(checkpoints)}] Checkpoint Exception: {e}")
    time.sleep(2)

  time.sleep(random.uniform(0.6, 1.2))

print("\nProcess ended.")


[!] Google decoder returned 429 (Too Many Requests) on items in 2025-07-20.
[710/914] Checkpoint Exception: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

[!] Google decoder returned 429 (Too Many Requests) on items in 2025-07-20.

[!] Google decoder returned 429 (Too Many Requests) on items in 2025-07-20.


KeyboardInterrupt: Interrupted by user

## 6. Pembersihan Akhir dan Ringkasan Dataset

Langkah terakhir untuk merapikan dataset:
1. Menghapuskan baris yang kosong atau memiliki judul ganda (duplikat).
2. Mengurutkan berita secara kronologis berdasarkan tanggal rilisnya.
3. Menyimpan file CSV final dan menampilkan 10 sampel data teratas sebagai preview.

In [ ]:
df_final = pd.DataFrame(records)

if not df_final.empty:
    df_final = df_final.dropna(subset=["title", "content", "published_at"]).drop_duplicates(subset=["title"])
    
    # 1. Normalize timezones
    df_final["published_at"] = pd.to_datetime(df_final["published_at"], utc=True).dt.tz_localize(None)
    df_final = df_final.sort_values(by="published_at").reset_index(drop=True)

    # 2. Filter exact 5-year bounds FIRST
    df_final = df_final[
        (df_final["published_at"] >= pd.Timestamp(START_DATE)) &
        (df_final["published_at"] < pd.Timestamp(END_DATE))
    ]
    
    # 3. Save the clean, bounded data to CSV
    df_final.to_csv(SAVE_PATH, index=False)

    print("=" * 60)
    print(f"Dataset compiled! Saved {len(df_final)} verified articles to: {SAVE_PATH}")
    print("=" * 60)
    
    preview = df_final[["published_at", "title", "source_domain"]].copy()
    preview["content_chars"] = df_final["content"].str.len()
    display(preview.head(10))
else:
    print("No records saved. Check network connection or query parameters.")

No records saved. Check network connection or query parameters.
